# SSL-AASIST Reproduction — Colab GPU Training

Runs the same pipeline you already validated locally, but on a free GPU.
Nothing here touches your laptop's disk — everything downloads/trains on
Google's servers. Checkpoints get saved to your Google Drive so they survive
a session disconnect; the dataset itself stays on Colab's local (ephemeral)
disk since it's fast to re-download here if a session resets.

**Before running**: Runtime menu → Change runtime type → Hardware accelerator → GPU (T4 is fine).


## 1. Confirm you actually got a GPU

In [2]:
%pip install torch

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
    --------------------------------------- 2.6/124.1 MB 12.6 MB/s eta 0:00:10
   - -------------------------------------- 5.0/124.1 MB 12.1 MB/s eta 0:00:10
   -- ------------------------------------- 7.6/124.1 MB 12.4 MB/s eta 0:00:10
   --- ------------------------------------ 11.0/124.1 MB 13.0 MB/s eta 0:00:09
   ---- ----------------------------------- 14.4/124.1 MB 13.5 MB/s eta 0:00:09
   ----- ---------------------------------- 17.6/124.1 MB 13.7 MB/s eta 0:00:08
   ------ --------------------------------- 20.2/124.1 MB 13.6 MB/s eta 0:00:08
   ------- -------------------------------- 23.1/124.1 MB 13.5 MB/s eta 0:00:08
   -------- -------------------------


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU attached — go to Runtime > Change runtime type > GPU, then Runtime > Restart session")


CUDA available: False
No GPU attached — go to Runtime > Change runtime type > GPU, then Runtime > Restart session


## 2. Get the code

Option A (recommended): clone your pushed GitHub repo — replace the URL below with yours.
Option B: if you haven't pushed yet, use the Colab file browser (left sidebar) to
upload a zip of your project, then unzip it instead of running the git clone cell.


In [ ]:
# --- Option A: clone from GitHub ---
REPO_URL = "https://github.com/AyeshaKODER/ssl-aasist-reproduction.git"  # <-- change this
!git clone {REPO_URL} project
%cd project


In [ ]:
# --- Option B: uploaded a zip instead? uncomment and run this, skip the clone cell above ---
# !unzip -q /content/ssl-aasist-reproduction.zip -d /content/
# %cd /content/ssl-aasist-reproduction


## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt


## 4. Mount Google Drive (for checkpoint persistence only)

The dataset stays local to this Colab session (fast, re-downloadable).
Only the small trained checkpoints get written to Drive, so a disconnect
doesn't lose your training progress.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = "/content/drive/MyDrive/ssl-aasist-runs"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)


## 5. Download ASVspoof2019 LA (into Colab's local disk, not your Drive/laptop)

This is the same 7.12GB LA.zip from before — but downloading server-to-server
here should be far faster than your home connection was.


In [ ]:
import os
os.makedirs("data", exist_ok=True)
%cd data

# Edinburgh DataShare direct download — if this specific link changes, get the
# current one from https://datashare.ed.ac.uk/handle/10283/3336 and paste it here
!wget -c "https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip" -O LA.zip

!unzip -q LA.zip
!rm LA.zip  # free the space immediately, we only need the extracted folder
%cd ..


## 6. Sanity check the extracted structure

In [ ]:
import os
expected = "data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
print("Protocol file found:", os.path.exists(expected))

train_flac_dir = "data/LA/ASVspoof2019_LA_train/flac"
print("Train audio files:", len(os.listdir(train_flac_dir)) if os.path.exists(train_flac_dir) else "MISSING")


## 7. Quick pipeline check — same subset run you already validated locally

This should finish in well under a minute on a T4, versus the ~2.6 hours/epoch
your CPU was projecting. If this looks sane, move to the full run below.


In [ ]:
!python src/train.py --config sinc_no_sa_no_da --subset_frac 0.05 --epochs 3 --seed 1 \
    --out_dir /content/drive/MyDrive/ssl-aasist-runs


## 8. Full training run — matches the paper's Section 6.3 exactly

100 epochs, real dataset, checkpoints saved straight to your Drive as they go.
If Colab disconnects mid-run, your last saved checkpoint is safe in Drive —
you'd resume by re-running from cell 1, then loading that checkpoint rather
than starting over (ask me for a resume flag if you hit this).


In [ ]:
!python src/train.py --config sinc_no_sa_no_da --epochs 100 --seed 1 --fp16 \
    --out_dir /content/drive/MyDrive/ssl-aasist-runs


## 9. Once training finishes — evaluate against the paper's Table 2

You'll need the ASVspoof2021 LA eval set + keys for this (same two files from
your earlier download — LA-keys-full.tar.gz and ASVspoof2021_LA_eval.tar.gz).
Download them here the same way as step 5 once you're ready for this step,
rather than up front.


In [ ]:
!python src/evaluate.py --config sinc_no_sa_no_da \
    --checkpoint /content/drive/MyDrive/ssl-aasist-runs/sinc_no_sa_no_da/seed1/best.pt \
    --eval_protocol data/ASVspoof2021_LA_eval/keys/<trial_metadata_filename>.txt \
    --eval_dir data/ASVspoof2021_LA_eval/flac
